# 3-D Benchmark — Prompt U-Net (multi-mode) vs nnInteractive

**Combined runner + results notebook.**

All configured Prompt-UNet **modes** are evaluated on the **exact same initial prompt** within each volume run,
enabling a fair within-prompt comparison (e.g. SSF vs IFL vs IFL+SSF).  
nnInteractive is run **once** per (volume, run) as a single-click baseline.

| Workflow |
|---|
| **Section 1** – Configure paths & parameters |
| **Section 2** – Run the benchmark *(skip if loading saved results)* |
| **Section 3** – Load results & analyse |
| **Section 4** – Export flat CSV |

---
## Section 1 — Configuration

In [ ]:
import sys, os

# ── Project root ──────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# ── Model paths ───────────────────────────────────────────────────────────────
P_UNET_MODELS = [
    os.path.join(PROJECT_ROOT, 'training', 'p_unet_315.keras'),
    # add more model paths to compare multiple checkpoints
]
NN_MODEL_DIR = None   # None = auto-download from HuggingFace Hub

# ── Dataset .npz paths (ds_handler format) ────────────────────────────────────
NPZ_PATHS = [
    os.path.join(PROJECT_ROOT, 'data', 'test_data', 'han_seg_ct.npz'),
    os.path.join(PROJECT_ROOT, 'data', 'test_data', 'han_seg_mri.npz'),
    os.path.join(PROJECT_ROOT, 'data', 'test_data', 'SegRap2023.npz'),
    os.path.join(PROJECT_ROOT, 'data', 'test_data', 'HCCTase_ceCT.npz'),
]

# ── Modes: ALL evaluated on the SAME initial prompt per run ───────────────────
# Supported: 'ssf', 'ifl', 'ifl_ssf', 'none'
MODES = ['ssf', 'ifl', 'ifl_ssf']

# ── Run parameters ────────────────────────────────────────────────────────────
RUNS_PER_VOL      = 5       # random prompts evaluated per volume
MODALITY          = None    # fallback modality if not stored in .npz
OUTPUT_THRESHOLD  = 0.5
SSF_THRESHOLD     = 0.40    # SSIM drop that triggers SSF refresh
GT_DICE_THRESHOLD = 0.65    # IFL Dice threshold for GT correction
BATCH_SIZE        = 6       # slices per GPU forward pass
WINDOW            = 10      # half-width for windowed Dice evaluation
MIN_PROMPT_PIXELS = 50      # minimum foreground pixels for prompt eligibility
NN_DEVICE         = None    # None = auto-detect (CUDA if available, else CPU)
MAX_VOLUMES       = None    # int (e.g. 3) for a quick test, None = evaluate all

# ── Output directory (pkl + json are saved here) ──────────────────────────────
OUTPUT_DIR = os.getcwd()

print('Configuration loaded.')
print(f'  Models : {[os.path.basename(m) for m in P_UNET_MODELS]}')
print(f'  Modes  : {MODES}  ← all run on the SAME prompt per run')
print(f'  npz    : {[os.path.basename(p) for p in NPZ_PATHS]}')
print(f'  Max Volumes: {MAX_VOLUMES if MAX_VOLUMES else "All"}')
print(f'  Output : {OUTPUT_DIR}')

---
## Section 2 — Run Benchmark

> ⚠️ **Skip this section** if you already have a `.pkl` file and only want to analyse results.

In [ ]:
from inference.ssf import RelativeSSIMStrategy
from evaluation.benchmark_nninteractive.benchmark_3d import run_benchmark

all_records = []

for p_model in P_UNET_MODELS:
    print(f"\n--- Running: {os.path.basename(p_model)} ---")
    records = run_benchmark(
        npz_paths         = NPZ_PATHS,
        p_unet_model      = p_model,
        nn_model_dir      = NN_MODEL_DIR,
        runs_per_vol      = RUNS_PER_VOL,
        modes             = MODES,          # ← list: all run on same prompt
        modality          = MODALITY,
        output_threshold  = OUTPUT_THRESHOLD,
        ssf_strategy      = RelativeSSIMStrategy(SSF_THRESHOLD),
        gt_dice_threshold = GT_DICE_THRESHOLD,
        batch_size        = BATCH_SIZE,
        window            = WINDOW,
        min_prompt_pixels = MIN_PROMPT_PIXELS,
        max_volumes       = MAX_VOLUMES,
        output_dir        = OUTPUT_DIR,
        nn_device         = NN_DEVICE,
        verbose           = True,
        return_predictions= False,
    )
    all_records.extend(records)

print(f"\nAll benchmark loops complete — {len(all_records)} total runs collected.")

---
## Section 3 — Load & Analyse Results

Either `all_records` from Section 2 is already in memory, or load an existing `.pkl` file below.

In [ ]:
import pickle, glob, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Option A: use results already in memory from Section 2 ────────────────────
# records = all_records   # already set above — nothing to do

# ── Option B: load the most-recent pkl from OUTPUT_DIR ───────────────────────
pkl_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'results_*.pkl')))
if not pkl_files:
    raise FileNotFoundError(f'No results .pkl found in {OUTPUT_DIR}')
LOAD_PATH = pkl_files[-1]   # ← change to a specific filename if needed
print(f'Loading: {LOAD_PATH}')
with open(LOAD_PATH, 'rb') as f:
    records = pickle.load(f)

print(f'Loaded {len(records)} runs.')
print(f'Modes evaluated : {records[0]["modes_evaluated"]}')
print(f'Datasets        : {sorted(set(r["dataset_name"] for r in records))}')

In [ ]:
# ── Build analysis DataFrames ─────────────────────────────────────────────────

MODES_EVALUATED = records[0].get('modes_evaluated', [])
DATASETS        = sorted(set(r['dataset_name'] for r in records))
WINDOW_CFG      = records[0].get('window', WINDOW if 'WINDOW' in dir() else 10)

# Top-level DataFrame — one row per (volume, run), nnInteractive columns
df = pd.DataFrame([{
    'volume_id'     : r['volume_id'],
    'pid'           : r['pid'],
    'run_idx'       : r['run_idx'],
    'dataset_name'  : r['dataset_name'],
    'modality'      : r['modality'],
    'p_unet_model'  : r['p_unet_model'],
    'prompt_axis'   : r['prompt_axis'],
    'prompt_idx'    : r['prompt_idx'],
    'selected_roi'  : r['selected_roi'],
    'nn_vol_dice'   : r['nn_vol_dice'],
    'nn_window_dice': r['nn_window_dice'],
    'nn_time_s'     : r['nn_time_s'],
} for r in records])

# Long-format DataFrame — one row per (volume, run, model)
rows_long = []
for r in records:
    base = {
        'volume_id'   : r['volume_id'],
        'pid'         : r['pid'],
        'run_idx'     : r['run_idx'],
        'dataset_name': r['dataset_name'],
        'modality'    : r['modality'],
        'prompt_axis' : r['prompt_axis'],
    }
    for mode, md in r.get('per_mode', {}).items():
        rows_long.append({
            **base,
            'model'             : f'P-UNet [{mode}]',
            'mode'              : mode,
            'vol_dice'          : md['vol_dice'],
            'window_dice'       : md['window_dice'],
            'time_s'            : md['time_s'],
            'num_user_interacts': md.get('num_user_interacts'),
            'num_slices'        : md.get('num_slices_evaluated'),
        })
    rows_long.append({
        **base,
        'model'             : 'nnInteractive',
        'mode'              : 'nnInteractive',
        'vol_dice'          : r['nn_vol_dice'],
        'window_dice'       : r['nn_window_dice'],
        'time_s'            : r['nn_time_s'],
        'num_user_interacts': None,
        'num_slices'        : None,
    })

df_long = pd.DataFrame(rows_long)
ALL_MODELS = MODES_EVALUATED + ['nnInteractive']

# Colour palette
PALETTE = {
    'ssf'          : '#4A90D9',
    'ifl'          : '#50C878',
    'ifl_ssf'      : '#9B59B6',
    'none'         : '#95A5A6',
    'nnInteractive': '#E87040',
}
DEFAULT_COLOR = '#888888'

print(f'Wide df  : {len(df)} rows')
print(f'Long df  : {len(df_long)} rows')
print(f'Modes    : {MODES_EVALUATED}')
print(f'Datasets : {DATASETS}')
df.head()

### 3.1 — Overall summary statistics

In [ ]:
def _fmt(series):
    s = pd.Series(series).dropna()
    if len(s) == 0:
        return 'N/A'
    return (f'{s.mean():.3f} ± {s.std():.3f}  '
            f'[med={s.median():.3f}  min={s.min():.3f}  max={s.max():.3f}  n={len(s)}]')

print(f"{'='*76}")
print(f"  BENCHMARK SUMMARY   n_runs={len(records)}  modes={MODES_EVALUATED}")
print(f"{'='*76}")

for mode in MODES_EVALUATED:
    m_rows = df_long[df_long['mode'] == mode]
    print(f"\n  P-UNet [{mode}]")
    print(f"    Vol Dice     : {_fmt(m_rows['vol_dice'])}")
    print(f"    Window Dice  : {_fmt(m_rows['window_dice'])}")
    print(f"    Time (s)     : {_fmt(m_rows['time_s'])}")
    ifl_col = m_rows['num_user_interacts'].dropna()
    if len(ifl_col):
        print(f"    IFL interacts: {_fmt(ifl_col)}")

nn_rows = df_long[df_long['mode'] == 'nnInteractive']
print(f"\n  nnInteractive  (initial prompt only)")
print(f"    Vol Dice     : {_fmt(nn_rows['vol_dice'])}")
print(f"    Window Dice  : {_fmt(nn_rows['window_dice'])}")
print(f"    Time (s)     : {_fmt(nn_rows['time_s'])}")
print(f"{'='*76}")

### 3.2 — Mode comparison boxplots (all modes + nnInteractive)

In [ ]:
model_labels = ALL_MODELS
colors       = [PALETTE.get(m, DEFAULT_COLOR) for m in model_labels]

fig, axes = plt.subplots(1, 2, figsize=(max(9, len(model_labels) * 1.8), 5), sharey=True)

for metric_col, ax, title in [
    ('vol_dice',    axes[0], 'Volumetric Dice'),
    ('window_dice', axes[1], f'Window Dice (±{WINDOW_CFG} slices)'),
]:
    data_list = [
        df_long[df_long['mode'] == m][metric_col].dropna().tolist()
        for m in model_labels
    ]
    bp = ax.boxplot(
        data_list,
        tick_labels  = [m.replace('_', '\n') for m in model_labels],
        widths       = 0.55,
        patch_artist = True,
        boxprops     = dict(linewidth=1.2),
        medianprops  = dict(color='white', linewidth=2.2),
        whiskerprops = dict(linewidth=1),
        capprops     = dict(linewidth=1),
        flierprops   = dict(marker='o', markersize=3.5, alpha=0.5),
    )
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    ax.set_title(title, fontsize=11, pad=8)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel('Dice')
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', labelsize=9)

fig.suptitle(
    f'Prompt U-Net Modes vs nnInteractive  (n={len(records)} runs, same prompt per run)',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dice_boxplots_all_modes.pdf'), bbox_inches='tight')
plt.show()
print('Saved: dice_boxplots_all_modes.pdf')

### 3.3 — Scatter: each P-UNet mode vs nnInteractive (per run)

In [ ]:
n_modes = len(MODES_EVALUATED)
fig, axes = plt.subplots(1, n_modes, figsize=(5 * n_modes, 5), sharey=True)
if n_modes == 1:
    axes = [axes]

for ax, mode in zip(axes, MODES_EVALUATED):
    mode_df = df_long[df_long['mode'] == mode][['volume_id', 'run_idx', 'vol_dice']].copy()
    merged  = mode_df.merge(
        df[['volume_id', 'run_idx', 'nn_vol_dice']],
        on=['volume_id', 'run_idx']
    )
    color = PALETTE.get(mode, DEFAULT_COLOR)
    ax.scatter(
        merged['vol_dice'], merged['nn_vol_dice'],
        alpha=0.6, s=28, color=color, edgecolors='none',
    )
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.5, label='Equal')
    ax.set_xlabel(f'P-UNet [{mode}]  Vol Dice', fontsize=10)
    ax.set_ylabel('nnInteractive  Vol Dice', fontsize=10)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(f'{mode}  (n={len(merged)})', fontsize=11)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

fig.suptitle('P-UNet Modes vs nnInteractive — Volumetric Dice Scatter', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'scatter_modes_vs_nn.pdf'), bbox_inches='tight')
plt.show()
print('Saved: scatter_modes_vs_nn.pdf')

### 3.4 — Per-dataset summary table

In [ ]:
rows = []
for ds in DATASETS:
    ds_df = df_long[df_long['dataset_name'] == ds]
    for model in ALL_MODELS:
        sub = ds_df[ds_df['mode'] == model]
        if len(sub) == 0:
            continue
        rows.append({
            'dataset'     : ds,
            'model'       : model,
            'n_runs'      : len(sub),
            'vol_mean'    : sub['vol_dice'].mean(),
            'vol_std'     : sub['vol_dice'].std(),
            'win_mean'    : sub['window_dice'].mean(),
            'win_std'     : sub['window_dice'].std(),
            'time_mean_s' : sub['time_s'].mean(),
            'time_std_s'  : sub['time_s'].std(),
        })

ds_table = pd.DataFrame(rows)
ds_table['Vol Dice']  = ds_table.apply(lambda r: f"{r['vol_mean']:.3f} ±{r['vol_std']:.3f}", axis=1)
ds_table['Win Dice']  = ds_table.apply(lambda r: f"{r['win_mean']:.3f} ±{r['win_std']:.3f}", axis=1)
ds_table['Time (s)']  = ds_table.apply(lambda r: f"{r['time_mean_s']:.1f} ±{r['time_std_s']:.1f}", axis=1)

display_cols = ['dataset', 'model', 'n_runs', 'Vol Dice', 'Win Dice', 'Time (s)']
display(ds_table[display_cols].set_index(['dataset', 'model']))

### 3.5 — Per-dataset Dice bar charts

In [ ]:
n_ds = len(DATASETS)
fig, axes = plt.subplots(n_ds, 2, figsize=(12, 4.2 * n_ds), squeeze=False)

x      = np.arange(len(ALL_MODELS))
width  = 0.6
bcolors = [PALETTE.get(m, DEFAULT_COLOR) for m in ALL_MODELS]

for row, ds in enumerate(DATASETS):
    ds_df = df_long[df_long['dataset_name'] == ds]

    for col, (metric, title) in enumerate([
        ('vol_dice',    'Volumetric Dice'),
        ('window_dice', f'Window Dice (±{WINDOW_CFG} slices)'),
    ]):
        ax = axes[row][col]
        means = [ds_df[ds_df['mode'] == m][metric].mean() for m in ALL_MODELS]
        stds  = [ds_df[ds_df['mode'] == m][metric].std()  for m in ALL_MODELS]
        bars  = ax.bar(
            x, means, width, yerr=stds, color=bcolors,
            edgecolor='white', linewidth=0.8,
            capsize=4, error_kw={'linewidth': 1.1}
        )
        ax.set_xticks(x)
        ax.set_xticklabels([m.replace('_', '\n') for m in ALL_MODELS], fontsize=9)
        ax.set_ylim(0, 1.1)
        ax.set_ylabel('Dice')
        ax.set_title(f'{ds}  —  {title}', fontsize=10)
        ax.grid(axis='y', alpha=0.3)
        # annotate bars
        for bar, mean in zip(bars, means):
            if not np.isnan(mean):
                ax.text(
                    bar.get_x() + bar.get_width() / 2, mean + 0.02,
                    f'{mean:.3f}', ha='center', va='bottom', fontsize=7.5
                )

fig.suptitle('Per-Dataset Dice by Mode', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dice_per_dataset_bars.pdf'), bbox_inches='tight')
plt.show()
print('Saved: dice_per_dataset_bars.pdf')

### 3.6 — Per-dataset Dice heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(max(10, len(ALL_MODELS) * 1.6), max(4, len(DATASETS) * 0.9)))

for ax, (metric, title) in zip(axes, [
    ('vol_dice',    'Volumetric Dice'),
    ('window_dice', f'Window Dice (±{WINDOW_CFG})'),
]):
    matrix = np.array([
        [
            df_long[(df_long['dataset_name'] == ds) & (df_long['mode'] == m)][metric].mean()
            for m in ALL_MODELS
        ]
        for ds in DATASETS
    ])
    im = ax.imshow(matrix, vmin=0, vmax=1, cmap='RdYlGn', aspect='auto')
    ax.set_xticks(range(len(ALL_MODELS)))
    ax.set_xticklabels([m.replace('_', '\n') for m in ALL_MODELS], fontsize=9)
    ax.set_yticks(range(len(DATASETS)))
    ax.set_yticklabels(DATASETS, fontsize=9)
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax)
    for i in range(len(DATASETS)):
        for j in range(len(ALL_MODELS)):
            v = matrix[i, j]
            if not np.isnan(v):
                ax.text(
                    j, i, f'{v:.3f}', ha='center', va='center',
                    color='black' if 0.2 < v < 0.8 else 'white', fontsize=8
                )

fig.suptitle('Per-Dataset Dice Heatmaps', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dice_heatmap_per_dataset.pdf'), bbox_inches='tight')
plt.show()
print('Saved: dice_heatmap_per_dataset.pdf')

### 3.7 — Inference timing analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# ── Left: average inference time per model (overall) ──────────────────────────
ax = axes[0]
x  = np.arange(len(ALL_MODELS))
means = [df_long[df_long['mode'] == m]['time_s'].mean() for m in ALL_MODELS]
stds  = [df_long[df_long['mode'] == m]['time_s'].std()  for m in ALL_MODELS]
bc    = [PALETTE.get(m, DEFAULT_COLOR) for m in ALL_MODELS]

bars = ax.bar(
    x, means, 0.6, yerr=stds, color=bc,
    edgecolor='white', linewidth=0.8, capsize=5
)
ax.set_xticks(x)
ax.set_xticklabels([m.replace('_', '\n') for m in ALL_MODELS], fontsize=9)
ax.set_ylabel('Inference Time (s)')
ax.set_title('Average Inference Time by Mode (all datasets)', fontsize=11)
ax.grid(axis='y', alpha=0.3)
for bar, mean in zip(bars, means):
    ax.text(
        bar.get_x() + bar.get_width() / 2, mean + stds[list(means).index(mean)] + 0.5,
        f'{mean:.1f}s', ha='center', va='bottom', fontsize=8
    )

# ── Right: per-dataset timing heatmap ─────────────────────────────────────────
ax = axes[1]
timing_matrix = np.array([
    [df_long[(df_long['dataset_name'] == ds) & (df_long['mode'] == m)]['time_s'].mean()
     for m in ALL_MODELS]
    for ds in DATASETS
])
im = ax.imshow(timing_matrix, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(ALL_MODELS)))
ax.set_xticklabels([m.replace('_', '\n') for m in ALL_MODELS], fontsize=9)
ax.set_yticks(range(len(DATASETS)))
ax.set_yticklabels(DATASETS, fontsize=9)
ax.set_title('Inference Time Heatmap (s)', fontsize=11)
plt.colorbar(im, ax=ax, label='seconds')
vmax = np.nanmax(timing_matrix)
for i in range(len(DATASETS)):
    for j in range(len(ALL_MODELS)):
        v = timing_matrix[i, j]
        if not np.isnan(v):
            ax.text(
                j, i, f'{v:.1f}', ha='center', va='center',
                color='black' if v < 0.7 * vmax else 'white', fontsize=8
            )

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'timing_analysis.pdf'), bbox_inches='tight')
plt.show()
print('Saved: timing_analysis.pdf')

### 3.8 — IFL interaction distribution  *(IFL modes only)*

In [ ]:
ifl_modes = [m for m in MODES_EVALUATED if 'ifl' in m]

if not ifl_modes:
    print('No IFL modes in this benchmark.')
else:
    fig, axes = plt.subplots(1, len(ifl_modes), figsize=(5.5 * len(ifl_modes), 4), squeeze=False)

    for ax, mode in zip(axes[0], ifl_modes):
        counts = (
            df_long[(df_long['mode'] == mode) & df_long['num_user_interacts'].notna()]
            ['num_user_interacts'].astype(int)
        )
        if counts.empty:
            ax.set_title(f'{mode}: no interaction data')
            continue
        bins  = range(counts.min(), counts.max() + 2)
        color = PALETTE.get(mode, DEFAULT_COLOR)
        ax.hist(counts, bins=bins, align='left', rwidth=0.72, color=color, edgecolor='white')
        ax.set_xlabel('Interactions (incl. initial prompt)', fontsize=10)
        ax.set_ylabel('Runs', fontsize=10)
        ax.set_title(f'[{mode}]  mean={counts.mean():.1f}  median={counts.median():.0f}', fontsize=11)
        ax.grid(axis='y', alpha=0.3)

    plt.suptitle('IFL Interaction Count Distribution', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'ifl_interactions.pdf'), bbox_inches='tight')
    plt.show()
    print('Saved: ifl_interactions.pdf')

### 3.9 — Dice by prompt axis

In [ ]:
axis_labels   = {0: 'Axial (0)', 1: 'Coronal (1)', 2: 'Sagittal (2)'}
present_axes  = sorted(df_long['prompt_axis'].dropna().unique())
n_pax         = len(present_axes)

fig, axes = plt.subplots(1, len(ALL_MODELS), figsize=(3.5 * len(ALL_MODELS), 4.5), sharey=True)
if len(ALL_MODELS) == 1:
    axes = [axes]

for ax, model in zip(axes, ALL_MODELS):
    sub     = df_long[df_long['mode'] == model]
    grouped = [sub[sub['prompt_axis'] == a]['vol_dice'].dropna().tolist() for a in present_axes]
    bp = ax.boxplot(
        grouped,
        tick_labels  = [axis_labels.get(a, str(a)) for a in present_axes],
        patch_artist = True,
        widths       = 0.5,
        boxprops     = dict(facecolor=PALETTE.get(model, DEFAULT_COLOR), linewidth=1.2),
        medianprops  = dict(color='white', linewidth=2),
        whiskerprops = dict(linewidth=1),
        capprops     = dict(linewidth=1),
    )
    ax.set_title(model.replace('_', '\n'), fontsize=10)
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel('Vol Dice')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Volumetric Dice by Prompt Axis', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dice_by_prompt_axis.pdf'), bbox_inches='tight')
plt.show()
print('Saved: dice_by_prompt_axis.pdf')

---
## Section 4 — Export flat CSV

In [ ]:
csv_rows = []
for r in records:
    row = {
        'volume_id'     : r['volume_id'],
        'pid'           : r['pid'],
        'run_idx'       : r['run_idx'],
        'dataset_name'  : r['dataset_name'],
        'modality'      : r['modality'],
        'p_unet_model'  : r['p_unet_model'],
        'prompt_axis'   : r['prompt_axis'],
        'prompt_idx'    : r['prompt_idx'],
        'selected_roi'  : r['selected_roi'],
        'nn_vol_dice'   : r['nn_vol_dice'],
        'nn_window_dice': r['nn_window_dice'],
        'nn_time_s'     : r['nn_time_s'],
    }
    for mode, md in r.get('per_mode', {}).items():
        row[f'{mode}_vol_dice']           = md['vol_dice']
        row[f'{mode}_window_dice']        = md['window_dice']
        row[f'{mode}_time_s']             = md['time_s']
        row[f'{mode}_num_slices']         = md.get('num_slices_evaluated')
        row[f'{mode}_num_user_interacts'] = md.get('num_user_interacts')
    csv_rows.append(row)

df_export = pd.DataFrame(csv_rows)
csv_path  = os.path.join(OUTPUT_DIR, 'results_flat.csv')
df_export.to_csv(csv_path, index=False)
print(f'Exported {len(df_export)} rows → {csv_path}')
df_export.head()